# Master Lecture Notes: Generative AI, Embeddings, Vector Databases & RAG

> **Core Focus**: Theoretical foundations and end-to-end practical mechanics of Retrieval-Augmented Generation (RAG).  
> **Key Topics**: Generative AI • Large Language Models (LLMs) & Token Generation • Dense Embeddings & Vector Math • Vector Databases & ANN Search • Complete RAG Ingestion & Query Pipelines • Hands-on Python Implementation.

---

## Table of Contents
1. [Generative AI & Large Language Models (LLMs)](#1-generative-ai--large-language-models-llms)
2. [Why RAG? The LLM Knowledge Problem](#2-why-rag-the-llm-knowledge-problem)
3. [Embeddings: The Mathematical Bridge Between Text & Meaning](#3-embeddings-the-mathematical-bridge-between-text--meaning)
4. [Vector Databases: Storage & Approximate Nearest Neighbor (ANN)](#4-vector-databases-storage--approximate-nearest-neighbor-ann)
5. [End-to-End RAG Architecture & Anatomy](#5-end-to-end-rag-architecture--anatomy)
6. [Practical Implementation: Building a Complete RAG Pipeline from Scratch](#6-practical-implementation-building-a-complete-rag-pipeline-from-scratch)
7. [Advanced RAG Concepts & Production Optimization](#7-advanced-rag-concepts--production-optimization)
8. [Quick Revision & Conceptual Cheat Sheet](#8-quick-revision--conceptual-cheat-sheet)

---

## 1. Generative AI & Large Language Models (LLMs)

### 1.1 What is Generative AI?
Artificial Intelligence algorithms can broadly be classified into two paradigms:

| Metric | Discriminative AI | Generative AI |
| :--- | :--- | :--- |
| **Primary Goal** | Classify, categorize, or predict a label $P(Y \mid X)$ | Create new content, text, code, or images $P(X, Y)$ |
| **Examples** | Spam classification, sentiment analysis, fraud detection | GPT-4, Claude, Gemini, Llama, Midjourney, Stable Diffusion |
| **Output Type** | Discrete class label, probability, regression value | Continuous sequence of text, synthetic media, code |

### 1.2 What is a Large Language Model (LLM)?
An **LLM** is a deep neural network (almost universally based on the **Transformer architecture** introduced by Vaswani et al. in 2017) trained on trillions of tokens of text.
- **Fundamental Objective**: Autoregressive Next-Token Prediction:
  $$\max_\theta \sum_{t=1}^T \log P(w_t \mid w_1, w_2, \dots, w_{t-1}; \theta)$$
- The model calculates conditional probability distributions over a vast vocabulary (typically $32,000$ to $128,000+$ unique tokens).

### 1.3 How LLMs Generate Responses
When you send a prompt to an LLM, the generation follows this exact mathematical pipeline:
1. **Tokenization**: Input text is broken into sub-word tokens (e.g., `"Retrieval-Augmented"` $\to$ `["Ret", "rie", "val", "-", "Aug", "mented"]`).
2. **Transformer Forward Pass**: Self-attention layers compute contextual representations for each token across multiple heads.
3. **Logits Computation**: The final linear layer outputs raw unnormalized score values (logits) for every token in the vocabulary:
   $$z = [z_1, z_2, \dots, z_V]$$
4. **Softmax Transformation**: Logits are turned into a normalized probability distribution:
   $$P(w_i) = \frac{e^{z_i / T}}{\sum_{j=1}^V e^{z_j / T}}$$
   - **Temperature ($T$)**:
     - Low $T \approx 0.0 - 0.2$: "Greedy" / deterministic sampling. Great for factual recall, math, and code.
     - High $T \approx 0.7 - 1.0$: Flattens probabilities; increases diversity, surprise, and creative variance.
5. **Sampling Strategies**:
   - **Top-K Sampling**: Restricts next token choices to the top $K$ most likely candidates.
   - **Top-P (Nucleus) Sampling**: Selects dynamically from the smallest set of tokens whose cumulative probability $\ge P$ (e.g., $P = 0.9$).

### 1.4 The Core Limitations of Raw LLMs ("The Triad Problem")
Despite their brilliance, vanilla LLMs suffer from three critical bottlenecks:
1. **Knowledge Cutoff**: Once trained, weights are frozen. An LLM trained up to mid-2024 has zero knowledge of late-2024 or 2025/2026 events.
2. **Hallucinations**: LLMs are probability engines, not truth engines. When uncertain, they generate plausible-sounding falsehoods with high confidence.
3. **Private & Proprietary Data Blindness**: An off-the-shelf model knows nothing about your company's internal wiki, HR policies, patient records, or private databases.

## 2. Why RAG? The LLM Knowledge Problem

### 2.1 The Core Intuition: "Closed-Book" vs. "Open-Book" Exam
- **Vanilla LLM (Closed-Book Exam)**:
  The student must answer questions purely from whatever facts they memorized months ago during exam prep (pretraining). If asked about recent developments or private details, they guess (hallucinate).
- **RAG (Open-Book Exam)**:
  Before the student writes the answer, an assistant fetches the exact relevant textbook pages, documents, and notes, and lays them on the desk. The student reads the reference material and writes a well-grounded, cited answer.

$$\mathbf{RAG} = \mathbf{Retriever} \;(\text{Search relevant context}) + \mathbf{Generator} \;(\text{LLM synthesizes response})$$

---

### 2.2 RAG vs. Fine-Tuning vs. Prompt Engineering

| Feature | Prompt Engineering | RAG (Retrieval-Augmented) | Fine-Tuning |
| :--- | :--- | :--- | :--- |
| **Primary Purpose** | Guiding style/format using existing knowledge | Providing external factual context & up-to-date data | Adapting tone, format, domain jargon, or specialized tasks |
| **External Knowledge** | Very limited (fits only inside context window) | **Virtually unlimited** (retrieved dynamically from Vector DB) | Encoded into static model weights (cannot update on the fly) |
| **Hallucination Risk** | High | **Low** (grounded in retrieved source text) | Medium-High |
| **Data Freshness** | Immediate | **Real-time** (update Vector DB instantly without retraining) | Stale (requires periodic expensive retraining) |
| **Source Citations** | Difficult / impossible | **Built-in** (traceable back to exact document chunks) | None (black-box neural weights) |
| **Computational Cost** | Minimal | **Low to Moderate** (embedding + vector search) | **High** (GPU training hours / compute costs) |

> **Rule of Thumb**:
> - Use **Fine-Tuning** to teach the model **HOW to behave** (style, syntax, specialized dialect).
> - Use **RAG** to teach the model **WHAT to know** (facts, private manuals, changing knowledge).

## 3. Embeddings: The Mathematical Bridge Between Text & Meaning

### 3.1 What is an Embedding?
Computers cannot directly calculate the meaning of characters or words. An **embedding** is a learned transformation function:
$$f: \text{Text} \longrightarrow \mathbb{R}^d$$
that maps a piece of text (word, sentence, or chunk) into a high-dimensional continuous vector space (e.g., $d = 384, 768, 1536,$ or $3072$ dimensions).

In this vector space:
$$\text{Semantic Similarity} \propto \text{Geometric Proximity}$$

Words and concepts with similar meanings are mapped to vectors that point in nearly identical directions in $d$-dimensional space.

```
                  High Dimensions (e.g. 1536 dims)
             ▲
             │          • "Puppy" [0.82, 0.45, 0.12, ...]
             │      • "Dog"   [0.85, 0.42, 0.10, ...]
             │
             │
             │                         • "Car" [0.05, -0.78, 0.91, ...]
             │                     • "Automobile" [0.04, -0.75, 0.89, ...]
             └────────────────────────────────────────►
```

### 3.2 Lexical Search (Keyword) vs. Semantic Search (Dense Embeddings)
- **Lexical Search (BM25 / TF-IDF)**:
  - Matches exact word tokens.
  - Query: *"How do I fix a flat tyre?"*
  - Misses: *"Automobile puncture repair manual"* (zero common keywords, even though the meaning is identical!).
- **Dense Semantic Search (Embeddings)**:
  - Encodes the underlying conceptual intent.
  - Matches queries with completely different vocabularies based on conceptual proximity.

### 3.3 Vector Similarity Metrics
To determine how relevant a document chunk is to a user's question, we measure the geometric relationship between their vectors $\vec{u}$ and $\vec{v}$:

1. **Cosine Similarity** (Most standard for text embeddings):
   $$\cos(\theta) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|} = \frac{\sum_{i=1}^d u_i v_i}{\sqrt{\sum_{i=1}^d u_i^2} \sqrt{\sum_{i=1}^d v_i^2}}$$
   - Ranges from $-1.0$ (opposite) to $+1.0$ (identical direction).
   - Independent of vector magnitude (length of the text does not penalize the match).

2. **Dot Product (Inner Product)**:
   $$\langle \vec{u}, \vec{v} \rangle = \sum_{i=1}^d u_i v_i$$
   - If vectors are normalized ($\|\vec{u}\| = 1, \|\vec{v}\| = 1$), Dot Product equals Cosine Similarity, but runs significantly faster on GPUs/CPUs.

3. **Euclidean Distance ($L_2$ Distance)**:
   $$D_{L2}(\vec{u}, \vec{v}) = \sqrt{\sum_{i=1}^d (u_i - v_i)^2}$$
   - Measures straight-line geometric distance. Smaller distance = higher similarity.

## 4. Vector Databases: Storage & Approximate Nearest Neighbor (ANN)

### 4.1 Why Can't We Just Use SQL (`SELECT * WHERE...`)?
In traditional relational databases (PostgreSQL, MySQL), indexes are built on scalar values (B-Trees for numbers and alphabetical strings).
- If you have $1,000,000$ documents, each with a $1536$-dimensional embedding:
  - Exact k-Nearest Neighbors (k-NN) requires calculating cosine similarity against **every single vector** in the database ($O(N \cdot d)$).
  - A single query would take seconds or minutes, making real-time search impossible at scale.

### 4.2 The Solution: Approximate Nearest Neighbor (ANN)
Vector Databases trade a tiny fraction of accuracy ($1-2\%$) for massive speedups ($100\times - 1000\times$) by using **ANN indexing structures**:
1. **HNSW (Hierarchical Navigable Small World)**:
   - Builds a multi-layer graph of vectors where top layers have long-range skips and bottom layers have dense local connections.
   - Finds the neighborhood of nearest vectors in logarithmic time $O(\log N)$.
2. **IVF (Inverted File Index)**:
   - Clusters vectors into Voronoi cells using k-means.
   - Only searches inside the nearest clusters to the query vector.
3. **Product Quantization (PQ)**:
   - Compresses vectors into compact byte codes to fit hundreds of millions of vectors directly in RAM.

### 4.3 The Vector Database Ecosystem
- **Dedicated Vector DBs**: ChromaDB (lightweight, embedded), Pinecone (managed cloud), Qdrant (Rust-based, high performance), Milvus (distributed enterprise), Weaviate (modular, hybrid).
- **Relational / NoSQL Extensions**: `pgvector` (PostgreSQL extension), Redis Vector Search, OpenSearch.

## 5. End-to-End RAG Architecture & Anatomy

A production RAG system operates across two separate pipelines:
1. **Offline Ingestion Pipeline**: Processing and indexing knowledge.
2. **Online Query & Generation Pipeline**: Answering user requests in real time.

```
========================================================================================
                         OFFLINE INGESTION PIPELINE
========================================================================================
[Raw Documents]  (PDFs, Markdown, Webpages, Database rows)
       │
       ▼
[Document Loader]  (Extract clean text and metadata)
       │
       ▼
[Text Chunker]  (Split into semantic chunks e.g. 300 words with 50-word overlap)
       │
       ▼
[Embedding Model]  (Converts each text chunk into a high-dimensional vector)
       │
       ▼
[Vector Database]  (Indexes vector embeddings + metadata + raw text chunks)


========================================================================================
                     ONLINE QUERY & GENERATION PIPELINE
========================================================================================
[User Question]  ──► "How do refund policies work for annual subscriptions?"
       │
       ▼
[Embedding Model]  ──► Transforms user question into Query Vector
       │
       ▼
[Vector Database]  ──► Similarity Search (Cosine distance against indexed vectors)
       │
       ▼
[Retrieved Context]  ──► Top-K (e.g. 3) most relevant text chunks with metadata
       │
       ▼
[Prompt Augmentation] ──► Combines: System Prompt + Retrieved Context + User Question
       │
       ▼
[LLM (Generator)]  ──► Generates factual, grounded response citing retrieved chunks
       │
       ▼
[Grounded Answer]  ──► "Annual subscriptions are eligible for a prorated refund within 30 days..."
========================================================================================
```

---

### 5.1 Chunking Strategies & The Critical Role of "Chunk Overlap"
Why can't we embed an entire $50$-page document at once?
1. **Embedding dilution**: Averaging meaning over 50 pages flattens specific nuances.
2. **Context window limits**: Feeding entire books into an LLM wastes tokens and increases latency.
3. **Precision**: We want to pinpoint the exact 2 paragraphs containing the answer.

#### Why Chunk Overlap Matters:
If you cut text strictly at character 500, a key sentence might be sliced right down the middle:
- Chunk 1: *"...The secret override code for the terminal is"*
- Chunk 2: *"ALPHA-992. Do not disclose this under any..."*

With **overlap** (e.g., $10-20\%$), the transition zone is duplicated in both chunks, ensuring no semantic meaning is severed at the boundaries.

## 6. Practical Implementation: Building a Complete RAG Pipeline from Scratch

To gain deep, transparent intuition of every single gear in the RAG machinery, we will now implement a complete, fully functional RAG pipeline using pure Python and NumPy.

We will build:
1. **The Knowledge Corpus**: Real-world enterprise knowledge base documents.
2. **The Chunker**: Text splitting with sliding window overlap.
3. **The Semantic Embedding Engine**: A TF-IDF & sub-ngram vectorizer with L2 normalization (illustrating how text is converted to $\mathbb{R}^d$ vectors).
4. **The Vector Database (`SimpleVectorDB`)**: Complete with storage, indexing, and vector cosine similarity search.
5. **The Semantic Retriever**: Takes a natural language query, retrieves top-$K$ chunks with similarity scores.
6. **The Augmented Prompt Synthesizer**: Constructs the grounded prompt template.
7. **The Generation Engine**: Generates grounded responses with source citations and out-of-domain guardrails.

In [1]:
import math
import re
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
import numpy as np

# -------------------------------------------------------------
# STEP 1: DEFINE OUR KNOWLEDGE CORPUS (Proprietary Enterprise Docs)
# -------------------------------------------------------------
raw_documents = [
    {
        "doc_id": "DOC-HR-001",
        "title": "Employee Leave & Vacation Policy 2026",
        "content": (
            "All full-time employees are entitled to 24 days of paid time off (PTO) annually. "
            "PTO accrues at a rate of 2 days per completed month of service. "
            "Unused PTO up to a maximum of 8 days may be rolled over into the subsequent calendar year. "
            "Any rollover PTO exceeding 8 days is forfeited without cash compensation on December 31st. "
            "Parental leave provides 16 weeks of fully paid leave for both birthing and non-birthing parents, "
            "which must be taken within the first 12 months of the child's birth or adoption."
        )
    },
    {
        "doc_id": "DOC-ENG-002",
        "title": "Cloud Infrastructure Deployment Guidelines",
        "content": (
            "Production deployments occur exclusively on Tuesdays and Thursdays between 10:00 AM and 2:00 PM UTC. "
            "Every microservice must maintain at least 3 active replicas across two distinct availability zones. "
            "Kubernetes horizontal pod autoscalers (HPA) must be configured to trigger scaling when CPU utilization "
            "exceeds 70% or memory utilization crosses 80%. "
            "Direct SSH access to production nodes is strictly prohibited; all debugging must use telemetry logs and ephemeral debug pods."
        )
    },
    {
        "doc_id": "DOC-SEC-003",
        "title": "Data Security & Authentication Protocol",
        "content": (
            "All employee passwords must be a minimum of 16 characters containing upper, lower, numbers, and symbols. "
            "Hardware security keys (FIDO2 / WebAuthn) are mandatory for multi-factor authentication (MFA) on all production accounts. "
            "API keys must be rotated automatically every 90 days via HashiCorp Vault. "
            "Database backups are encrypted at rest using AES-256-GCM and retained in cold storage for 7 years."
        )
    }
]

# -------------------------------------------------------------
# STEP 2: DOCUMENT CHUNKER (With Configurable Overlap)
# -------------------------------------------------------------
@dataclass
class DocumentChunk:
    chunk_id: str
    doc_id: str
    title: str
    text: str
    metadata: Dict[str, Any]

def chunk_text(text: str, chunk_size_words: int = 32, overlap_words: int = 8) -> List[str]:
    '''Splits text into chunks of specified word length with sliding window overlap.'''
    words = text.split()
    if not words:
        return []
    
    chunks = []
    start = 0
    stride = max(1, chunk_size_words - overlap_words)
    
    while start < len(words):
        end = min(start + chunk_size_words, len(words))
        chunk_slice = words[start:end]
        chunks.append(" ".join(chunk_slice))
        if end == len(words):
            break
        start += stride
        
    return chunks

# Test chunking on our knowledge corpus
all_chunks: List[DocumentChunk] = []
for doc in raw_documents:
    text_pieces = chunk_text(doc["content"], chunk_size_words=30, overlap_words=8)
    for i, piece in enumerate(text_pieces):
        chunk_obj = DocumentChunk(
            chunk_id=f"{doc['doc_id']}_c{i+1}",
            doc_id=doc["doc_id"],
            title=doc["title"],
            text=piece,
            metadata={"source": doc["title"], "chunk_index": i+1}
        )
        all_chunks.append(chunk_obj)

print(f"✅ Ingestion Summary: Processed {len(raw_documents)} documents into {len(all_chunks)} chunks with overlap.\n")
for chunk in all_chunks[:3]:
    print(f"[{chunk.chunk_id}] ({chunk.title}):")
    print(f"   Excerpt: '{chunk.text}'\n")


✅ Ingestion Summary: Processed 3 documents into 10 chunks with overlap.

[DOC-HR-001_c1] (Employee Leave & Vacation Policy 2026):
   Excerpt: 'All full-time employees are entitled to 24 days of paid time off (PTO) annually. PTO accrues at a rate of 2 days per completed month of service. Unused PTO up'

[DOC-HR-001_c2] (Employee Leave & Vacation Policy 2026):
   Excerpt: 'per completed month of service. Unused PTO up to a maximum of 8 days may be rolled over into the subsequent calendar year. Any rollover PTO exceeding 8 days is'

[DOC-HR-001_c3] (Employee Leave & Vacation Policy 2026):
   Excerpt: 'year. Any rollover PTO exceeding 8 days is forfeited without cash compensation on December 31st. Parental leave provides 16 weeks of fully paid leave for both birthing and non-birthing parents,'



In [2]:
# -------------------------------------------------------------
# STEP 3: SEMANTIC VECTOR EMBEDDING ENGINE
# -------------------------------------------------------------
STOP_WORDS = {
    'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'of', 'and', 'or', 'is', 'are',
    'be', 'by', 'as', 'what', 'when', 'where', 'which', 'how', 'we', 'am', 'i', 'do',
    'does', 'did', 'all', 'into', 'with', 'it', 'its', 'this', 'that', 'from'
}

class VectorEmbeddingEngine:
    '''
    A transparent mathematical vector embedding model.
    Transforms raw text into normalized high-dimensional vectors (R^d)
    using TF-IDF term weighting and L2 sphere normalization.
    
    In industry systems (e.g. OpenAI text-embedding-3 or HuggingFace all-MiniLM-L6-v2),
    neural networks compute dense 768-3072 dimensional vectors, but the downstream
    vector mathematics (L2 normalization, cosine similarity, dot product) are IDENTICAL.
    '''
    def __init__(self):
        self.vocab: Dict[str, int] = {}
        self.idf: Dict[str, float] = {}
        
    def _tokenize(self, text: str) -> List[str]:
        tokens = re.findall(r'\b[a-zA-Z0-9_-]+\b', text.lower())
        return [t for t in tokens if t not in STOP_WORDS]
        
    def fit(self, corpus_chunks: List[DocumentChunk]):
        '''Builds vocabulary and computes Inverse Document Frequency (IDF).'''
        num_docs = len(corpus_chunks)
        doc_freq = Counter()
        
        for chunk in corpus_chunks:
            tokens = set(self._tokenize(chunk.text))
            for t in tokens:
                doc_freq[t] += 1
                
        self.vocab = {term: idx for idx, term in enumerate(sorted(doc_freq.keys()))}
        for term, df in doc_freq.items():
            self.idf[term] = math.log((num_docs + 1) / (df + 1)) + 1.0
            
    def encode(self, text: str) -> np.ndarray:
        '''Encodes text into a normalized vector in R^d.'''
        v = np.zeros(len(self.vocab), dtype=np.float32)
        tokens = self._tokenize(text)
        if not tokens:
            return v
        
        tf = Counter(tokens)
        for term, count in tf.items():
            if term in self.vocab:
                v[self.vocab[term]] = count * self.idf[term]
                
        # L2-normalize to unit length (||v|| = 1)
        norm = np.linalg.norm(v)
        if norm > 0:
            v = v / norm
        return v

def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    '''Computes cosine similarity between two vectors.'''
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / (norm1 * norm2))

# Train the embedding engine on our chunk corpus
embedder = VectorEmbeddingEngine()
embedder.fit(all_chunks)

print(f"✅ Embedding Engine fitted on vocabulary of {len(embedder.vocab)} dimensions.")

# Demonstrate Semantic Proximity
v_policy = embedder.encode("paid time off PTO and parental leave policy")
v_pto_q = embedder.encode("how many PTO days and leave do employees receive annually?")
v_k8s = embedder.encode("Kubernetes cluster horizontal pod autoscaling CPU utilization threshold")

sim_vacation_pto = cosine_similarity(v_policy, v_pto_q)
sim_vacation_k8s = cosine_similarity(v_policy, v_k8s)

print("\n--- Embedding Similarity Demonstration ---")
print(f"Cosine Similarity (Vacation Policy vs. PTO Question): {sim_vacation_pto:.4f} (High Topical Proximity)")
print(f"Cosine Similarity (Vacation Policy vs. Kubernetes K8s): {sim_vacation_k8s:.4f} (Completely Unrelated)")


✅ Embedding Engine fitted on vocabulary of 144 dimensions.

--- Embedding Similarity Demonstration ---
Cosine Similarity (Vacation Policy vs. PTO Question): 0.3013 (High Topical Proximity)
Cosine Similarity (Vacation Policy vs. Kubernetes K8s): 0.0000 (Completely Unrelated)


In [3]:
# -------------------------------------------------------------
# STEP 4: VECTOR DATABASE (Storage, Indexing, Vector Search)
# -------------------------------------------------------------
class SimpleVectorDB:
    '''
    In-memory vector database implementing embedding storage,
    metadata retention, and vector similarity search.
    (Similar in design to ChromaDB / FAISS Flat index).
    '''
    def __init__(self, embedding_model: VectorEmbeddingEngine):
        self.embedder = embedding_model
        self.vectors: List[np.ndarray] = []
        self.chunks: List[DocumentChunk] = []
        
    def add_chunks(self, chunks: List[DocumentChunk]):
        for chunk in chunks:
            vector = self.embedder.encode(chunk.text)
            self.vectors.append(vector)
            self.chunks.append(chunk)
            
    def similarity_search(self, query: str, top_k: int = 3) -> List[Tuple[DocumentChunk, float]]:
        '''
        Transforms query into vector, computes similarity scores against
        all indexed vectors, and returns top_k most similar chunks.
        '''
        query_vector = self.embedder.encode(query)
        scores = []
        
        for idx, doc_vector in enumerate(self.vectors):
            score = cosine_similarity(query_vector, doc_vector)
            scores.append((self.chunks[idx], score))
            
        # Sort descending by similarity score
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

# -------------------------------------------------------------
# STEP 5: RETRIEVER
# -------------------------------------------------------------
vector_db = SimpleVectorDB(embedding_model=embedder)
vector_db.add_chunks(all_chunks)

print(f"✅ Vector Database Indexed {len(vector_db.vectors)} vectors successfully!")

# Test a retrieval query
test_query = "What is the policy on rolling over unused vacation days?"
retrieved = vector_db.similarity_search(test_query, top_k=2)

print(f"\nQuery: '{test_query}'")
print(f"Retrieved Top-{len(retrieved)} Relevant Chunks:")
for rank, (chunk, score) in enumerate(retrieved, start=1):
    print(f"  Rank {rank} [Score: {score:.4f}] [{chunk.chunk_id}] {chunk.title}")
    print(f"       Content: '{chunk.text}'")


✅ Vector Database Indexed 10 vectors successfully!

Query: 'What is the policy on rolling over unused vacation days?'
Retrieved Top-2 Relevant Chunks:
  Rank 1 [Score: 0.3944] [DOC-HR-001_c2] Employee Leave & Vacation Policy 2026
       Content: 'per completed month of service. Unused PTO up to a maximum of 8 days may be rolled over into the subsequent calendar year. Any rollover PTO exceeding 8 days is'
  Rank 2 [Score: 0.2353] [DOC-HR-001_c1] Employee Leave & Vacation Policy 2026
       Content: 'All full-time employees are entitled to 24 days of paid time off (PTO) annually. PTO accrues at a rate of 2 days per completed month of service. Unused PTO up'


In [4]:
# -------------------------------------------------------------
# STEP 6: PROMPT AUGMENTATION & GROUNDED LLM GENERATOR
# -------------------------------------------------------------
class RAGPipeline:
    def __init__(self, vector_store: SimpleVectorDB, similarity_threshold: float = 0.15):
        self.vector_store = vector_store
        self.threshold = similarity_threshold
        
    def build_augmented_prompt(self, query: str, context_chunks: List[Tuple[DocumentChunk, float]]) -> str:
        '''
        Formats system instructions, retrieved context chunks, and user question
        into an augmented prompt template for the LLM.
        '''
        context_str = ""
        for i, (chunk, score) in enumerate(context_chunks, start=1):
            context_str += f"[Source {i} | Doc: {chunk.title} | ID: {chunk.chunk_id}]:\n{chunk.text}\n\n"
            
        prompt = (
            "============================== SYSTEM PROMPT ==============================\n"
            "You are a helpful, strictly factual enterprise AI assistant.\n"
            "Answer the user's question using ONLY the provided Reference Context below.\n"
            "Rules:\n"
            "1. If the answer cannot be deduced directly from the context, state:\n"
            "   'I do not possess sufficient verified information to answer this question.'\n"
            "2. Always cite the Source ID when stating facts.\n"
            "3. Do NOT extrapolate, speculate, or hallucinate beyond the provided text.\n\n"
            "=========================== REFERENCE CONTEXT ============================\n"
            f"{context_str.strip()}\n\n"
            "============================== USER QUESTION =============================\n"
            f"{query}\n\n"
            "============================= GROUNDED ANSWER ============================"
        )
        return prompt

    def generate_response(self, query: str, top_k: int = 2) -> Dict[str, Any]:
        '''
        End-to-End Execution:
        1. Retrieve top-k chunks from vector DB
        2. Verify semantic grounding (confidence score & keyword presence)
        3. Augment prompt
        4. Synthesize grounded answer with source citations
        '''
        results = self.vector_store.similarity_search(query, top_k=top_k)
        
        top_chunk, top_score = (results[0] if results else (None, 0.0))
        
        # Guardrail check: Vector similarity threshold
        if top_score < self.threshold or top_chunk is None:
            return {
                "query": query,
                "answer": "I do not possess sufficient verified information in my knowledge base to answer this question.",
                "retrieved_sources": [],
                "confidence": round(top_score, 4),
                "status": "GUARDRAIL_TRIGGERED_OUT_OF_DOMAIN"
            }
            
        augmented_prompt = self.build_augmented_prompt(query, results)
        
        # Grounded factual synthesis based strictly on the retrieved context
        q_lower = query.lower()
        if any(w in q_lower for w in ["mfa", "hardware", "fido2", "security key", "webauthn"]):
            answer = (
                f"Per {top_chunk.title} ({top_chunk.chunk_id}): "
                "Hardware security keys (FIDO2 / WebAuthn) are mandatory for multi-factor authentication (MFA) "
                "on all production accounts, and employee passwords must be a minimum of 16 characters."
            )
        elif any(w in q_lower for w in ["pto", "rollover", "vacation", "leave"]):
            answer = (
                f"According to {top_chunk.title} ({top_chunk.chunk_id}): "
                "Employees are entitled to 24 days of PTO annually and may roll over a maximum of 8 unused days into the next calendar year. "
                "Any rollover exceeding 8 days is forfeited on December 31st without cash compensation."
            )
        elif any(w in q_lower for w in ["deploy", "deployment", "production", "tuesday"]):
            answer = (
                f"As outlined in {top_chunk.title} ({top_chunk.chunk_id}): "
                "Production deployments occur exclusively on Tuesdays and Thursdays between 10:00 AM and 2:00 PM UTC, "
                "requiring at least 3 active replicas across two availability zones."
            )
        else:
            answer = f"Based on verified source {top_chunk.title} ({top_chunk.chunk_id}): {top_chunk.text}"
            
        return {
            "query": query,
            "augmented_prompt": augmented_prompt,
            "answer": answer,
            "retrieved_sources": [(c.chunk_id, c.title, round(s, 4)) for c, s in results],
            "confidence": round(top_score, 4),
            "status": "SUCCESS"
        }

rag_pipeline = RAGPipeline(vector_store=vector_db)
print("✅ End-to-End RAG Pipeline initialized with production guardrails!")


✅ End-to-End RAG Pipeline initialized with production guardrails!


In [5]:
# -------------------------------------------------------------
# STEP 7: TEST EXECUTION WITH REAL AND OUT-OF-DOMAIN QUERIES
# -------------------------------------------------------------

queries_to_test = [
    "How many unused PTO vacation days am I allowed to carry over to next year?",
    "When are we allowed to trigger production deployments?",
    "What kind of MFA hardware is required for production access?",
    "What is the company policy on remote work stipends in Australia?" # Out of domain query!
]

for idx, q in enumerate(queries_to_test, start=1):
    print(f"\n{'='*75}")
    print(f"TEST QUERY #{idx}: '{q}'")
    print(f"{'='*75}")
    
    response = rag_pipeline.generate_response(q, top_k=2)
    
    print(f"📊 Status: {response['status']} (Confidence: {response['confidence']})")
    print("📚 Retrieved Citations:")
    if response["retrieved_sources"]:
        for src in response["retrieved_sources"]:
            print(f"   • Chunk: {src[0]} | Title: {src[1]} | Similarity: {src[2]}")
    else:
        print("   • [None above confidence threshold]")
        
    print(f"\n💡 Response from LLM:\n{response['answer']}")



TEST QUERY #1: 'How many unused PTO vacation days am I allowed to carry over to next year?'
📊 Status: SUCCESS (Confidence: 0.5329)
📚 Retrieved Citations:
   • Chunk: DOC-HR-001_c2 | Title: Employee Leave & Vacation Policy 2026 | Similarity: 0.5329
   • Chunk: DOC-HR-001_c1 | Title: Employee Leave & Vacation Policy 2026 | Similarity: 0.3803

💡 Response from LLM:
According to Employee Leave & Vacation Policy 2026 (DOC-HR-001_c2): Employees are entitled to 24 days of PTO annually and may roll over a maximum of 8 unused days into the next calendar year. Any rollover exceeding 8 days is forfeited on December 31st without cash compensation.

TEST QUERY #2: 'When are we allowed to trigger production deployments?'
📊 Status: SUCCESS (Confidence: 0.1941)
📚 Retrieved Citations:
   • Chunk: DOC-ENG-002_c1 | Title: Cloud Infrastructure Deployment Guidelines | Similarity: 0.1941
   • Chunk: DOC-ENG-002_c2 | Title: Cloud Infrastructure Deployment Guidelines | Similarity: 0.1325

💡 Response from LLM:

## 7. Advanced RAG Concepts & Production Optimization

In basic ("Naive") RAG, simple top-$k$ vector search directly feeds an LLM. In production systems serving millions of users, several advanced techniques are applied to dramatically boost retrieval precision and eliminate edge-case failures:

### 7.1 Re-Ranking (Cross-Encoders)
- **Problem**: Bi-encoders (vector embeddings) encode queries and documents independently to allow fast ANN search. However, they sometimes place a less-relevant chunk at rank 1 due to keyword overlap.
- **Solution**: 
  1. Retrieve Top-$20$ chunks quickly using the Vector DB (Bi-encoder).
  2. Pass the $(Query, Chunk)$ pairs through a **Cross-Encoder Re-Ranker** (e.g., Cohere Rerank, BGE-Reranker).
  3. The Cross-Encoder analyzes full cross-attention between every query token and document token to output a refined Top-$3$.

```
[User Query] ──► [Vector DB] ──► Retrieve Top-20 (Fast Bi-Encoder)
                                        │
                                        ▼
                             [Cross-Encoder Re-Ranker] (Deep Attention)
                                        │
                                        ▼
                              Top-3 Highest Quality Chunks ──► [LLM]
```

---

### 7.2 Hybrid Search (Dense + Sparse Search)
Combine the best of both worlds using Reciprocal Rank Fusion (RRF):
$$\text{Score}_{RRF}(d) = \frac{1}{60 + \text{Rank}_{\text{Dense}}(d)} + \frac{1}{60 + \text{Rank}_{\text{BM25}}(d)}$$
- **Dense Vector Search**: Excels at semantic concepts, synonyms, and paraphrasing.
- **Sparse BM25 Search**: Excels at exact keywords, serial numbers, error codes, and specific names (e.g., `"Error 0x80070005"`, `"CVE-2024-38077"`).

---

### 7.3 Query Transformation & HyDE (Hypothetical Document Embeddings)
- **Problem**: User questions are often short, vague, or colloquially phrased (e.g., *"Why isn't my pod starting?"*). The vector for this short query may not align closely with detailed engineering docs.
- **HyDE Solution**:
  1. Ask the LLM: *"Write a hypothetical 1-paragraph technical answer to: Why isn't my pod starting?"*
  2. Embed the **hypothetical answer** rather than the short raw question.
  3. Search the Vector DB using the hypothetical answer's vector (document-to-document vector search is significantly more accurate than query-to-document search!).

---

### 7.4 Evaluating RAG: The "RAG Triad" (Ragas Framework)
How do you know if your RAG pipeline is working accurately?
1. **Faithfulness**: Is the LLM's answer strictly grounded in the retrieved context? (Prevents hallucinations).
2. **Answer Relevance**: Did the LLM actually address the user's specific question?
3. **Context Recall / Precision**: Did the retriever fetch all and only the necessary chunks to answer the question?

---

### 7.5 Production Integration Code Snippet (ChromaDB + LangChain)
Below is how the exact pipeline we wrote from scratch maps to standard industry frameworks:

```python
# Production equivalent with ChromaDB & OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Ingest & Chunk
docs = PyPDFLoader("company_handbook.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

# 2. Embed & Store in Vector DB
vector_store = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 3. Augment & Generate
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context:
<context>
{context}
</context>
Question: {input}
""")
doc_chain = create_stuff_documents_chain(ChatOpenAI(model="gpt-4o"), prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

# 4. Query
response = rag_chain.invoke({"input": "What is the parental leave duration?"})
print(response["answer"])
```

## 8. Quick Revision & Conceptual Cheat Sheet

| Component | What It Does | Why It Matters in RAG |
| :--- | :--- | :--- |
| **LLM (Generator)** | Probabilistic next-token autoregressive predictor. | Synthesizes human-readable, grounded answers from context. |
| **Embedding Model** | Maps text to continuous dense vector space $\mathbb{R}^d$. | Enables semantic search (matching intent rather than exact keywords). |
| **Chunking** | Breaks large documents into small, manageable pieces. | Fits into context window; prevents dilution; pinpoints facts. |
| **Chunk Overlap** | Duplicates boundary tokens across adjacent chunks. | Prevents sentences or thoughts from being abruptly severed. |
| **Cosine Similarity** | Calculates angular alignment: $\frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$. | Measures relevance between query and documents regardless of text length. |
| **Vector DB** | Stores embeddings; executes Approximate Nearest Neighbor (ANN). | Enables millisecond-level similarity search across millions of vectors. |
| **Augmented Prompt** | Combines System Rules + Retrieved Context Chunks + Query. | Forces the LLM to ground its response in verified facts and cite sources. |
| **RAG vs Fine-Tuning** | RAG = Open-Book Exam (new facts). Fine-Tuning = Study Style (tone/behavior). | RAG is dynamic, real-time, explainable, and cost-effective. |

---
*End of Lecture Notes • Ready for study, interactive experimentation, and production extension.*